## Задание 1

Для датафрейма `log` создайте столбец `source_type` по следующим правилам:

* **`organic`**: если источник `traffic_source` равен **Yandex** или **Google**;
* **`ad`**: для источников `traffic_source` со значением **paid** и **email**, если страна (`region`) — **Russia**;
* **`other`**: для источников `traffic_source` со значением **paid** и **email**, если страна (`region`) — **не Russia**;
* **Без изменений**: все остальные варианты исходного `traffic_source`.

In [1]:
import pandas as pd

In [4]:
df_logs = pd.read_csv('visit_log.csv', sep=';')

In [5]:
df_logs

,timestamp,visit_id,url,region,user_id,traffic_source
0,1549980692,e3b0c44298,https://host.ru/3c19b4ef7371864fa3,Russia,b1613cc09f,yandex
1,1549980704,6e340b9cff,https://host.ru/c8d9213a31839f9a3a,Russia,4c3ec14bee,direct
2,1549980715,96a296d224,https://host.ru/b8b58337d272ee7b15,Russia,a8c40697fb,yandex
3,1549980725,709e80c884,https://host.ru/b8b58337d272ee7b15,Russia,521ac1d6a0,yandex
4,1549980736,df3f619804,https://host.ru/b8b58337d272ee7b15,Russia,d7323c571c,yandex
...,...,...,...,...,...,...
18933,1550094288,57e5ba8560,https://host.ru/c2382eb3d6afc8d0f3,Belarus,98b19810d0,paid
18934,1550094296,6f9389ec1b,https://host.ru/f1eb4601740d627ab0,Russia,32ebb20c13,paid
18935,1550094308,e8cf2eb8e6,https://host.ru/a5dda93e70318570c0,Belarus,b85baa8c73,yandex
18936,1550094314,79530b9a67,https://host.ru/6fda01ec57f23abc9e,Russia,e154b06121,paid


In [10]:
def source(row):
    if (row.traffic_source == 'yandex') or (row.traffic_source == 'google'):
        return 'organic'
    if (row.traffic_source == 'paid') or (row.traffic_source == 'email'):
        if row.region == 'Russia':
            return 'ad'
        else:
            return 'other'
    return row.traffic_source

In [11]:
df_logs['source_type'] = df_logs.apply(source, axis=1)

In [12]:
df_logs

,timestamp,visit_id,url,region,user_id,traffic_source,source_type
0,1549980692,e3b0c44298,https://host.ru/3c19b4ef7371864fa3,Russia,b1613cc09f,yandex,organic
1,1549980704,6e340b9cff,https://host.ru/c8d9213a31839f9a3a,Russia,4c3ec14bee,direct,direct
2,1549980715,96a296d224,https://host.ru/b8b58337d272ee7b15,Russia,a8c40697fb,yandex,organic
3,1549980725,709e80c884,https://host.ru/b8b58337d272ee7b15,Russia,521ac1d6a0,yandex,organic
4,1549980736,df3f619804,https://host.ru/b8b58337d272ee7b15,Russia,d7323c571c,yandex,organic
...,...,...,...,...,...,...,...
18933,1550094288,57e5ba8560,https://host.ru/c2382eb3d6afc8d0f3,Belarus,98b19810d0,paid,other
18934,1550094296,6f9389ec1b,https://host.ru/f1eb4601740d627ab0,Russia,32ebb20c13,paid,ad
18935,1550094308,e8cf2eb8e6,https://host.ru/a5dda93e70318570c0,Belarus,b85baa8c73,yandex,organic
18936,1550094314,79530b9a67,https://host.ru/6fda01ec57f23abc9e,Russia,e154b06121,paid,ad


## Задание 2

В файле `URLs.txt` содержатся URL страниц новостного сайта. Вам нужно отфильтровать его по адресам страниц с текстами новостей. Известно, что шаблон страницы новостей имеет внутри URL конструкцию: `/`, затем **8 цифр**, затем **дефис**.

### Алгоритм решения:
1. **Чтение данных**: Прочитайте содержимое файла в датафрейм.
2. **Фильтрация**: Отфильтруйте страницы с текстом новостей, используя метод `.str.contains()` и регулярное выражение в соответствии с заданным шаблоном.

In [15]:
df_urls = pd.read_csv('URLs.txt', names=['url'])
regex = r"/\d{8}-"
df_urls[df_urls["url"].str.contains(regex, regex=True)]

,url
4,/politics/36188461-s-marta-zhizn-rossiyan-susc...
5,/world/36007585-tramp-pridumal-kak-reshit-ukra...
6,/science/36157853-nasa-sobiraet-ekstrennuyu-pr...
7,/video/36001498-poyavilis-pervye-podrobnosti-g...
8,/world/36007585-tramp-pridumal-kak-reshit-ukra...
...,...
89,/cis/35984145-kreml-prokommentiroval-soobschen...
90,/video/36071019-olimpiyskie-obekty-rio-prevrat...
91,/science/36151301-nazvano-posledstvie-zloupotr...
92,/incidents/36027330-vospitatelnitsu-zatravili-...


## Задание 3

Используйте файл с оценками фильмов `ml-latest-small/ratings.csv.` Посчитайте среднее время жизни пользователей, которые выставили более 100 оценок. Под временем жизни понимается разница между максимальным и минимальным значениями столбца `timestamp` для данного значения `userId`.

In [16]:
df_ratings = pd.read_csv('ratings.csv')

In [19]:
df_user_stats = df_ratings.groupby('userId').agg(
    ratings_count=('rating', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
)

In [20]:
filtered_users =df_user_stats[df_user_stats['ratings_count'] > 100]

In [21]:
filtered_users['lifetime'] = filtered_users['max_timestamp'] - filtered_users['min_timestamp']

In [23]:
filtered_users

,ratings_count,min_timestamp,max_timestamp,lifetime
userId,,,,
4,204,949778714,949982274,203560
8,116,1154389340,1154474527,85187
15,1700,997937239,1469330735,471393496
17,363,1127468587,1127476640,8053
19,423,855190091,855195373,5282
...,...,...,...,...
656,128,986240991,986244044,3053
659,142,834598040,866207451,31609411
664,519,1343731283,1441911722,98180439


In [24]:
filtered_users['lifetime'].mean()

np.float64(40080507.4496124)

## Задание 4

Дана статистика услуг перевозок клиентов компании по типам см. файл `Python_13_join.ipynb`

Нужно сформировать две таблицы:

* таблицу с тремя типами выручки для каждого `client_id` без указания адреса клиент;
* аналогичную таблицу по типам выручки с указанием адреса клиента;

In [25]:
rzd = pd.DataFrame(
    {
        'client_id': [111, 112, 113, 114, 115],
        'rzd_revenue': [1093, 2810, 10283, 5774, 981]
    }
)
rzd

,client_id,rzd_revenue
0,111,1093
1,112,2810
2,113,10283
3,114,5774
4,115,981


In [26]:
auto = pd.DataFrame(
    {
        'client_id': [113, 114, 115, 116, 117],
        'auto_revenue': [57483, 83, 912, 4834, 98]
    }
)
auto

,client_id,auto_revenue
0,113,57483
1,114,83
2,115,912
3,116,4834
4,117,98


In [27]:
air = pd.DataFrame(
    {
        'client_id': [115, 116, 117, 118],
        'air_revenue': [81, 4, 13, 173]
    }
)
air

,client_id,air_revenue
0,115,81
1,116,4
2,117,13
3,118,173


In [28]:
client_base = pd.DataFrame(
    {
        'client_id': [111, 112, 113, 114, 115, 116, 117, 118],
        'address': ['Комсомольская 4', 'Энтузиастов 8а', 'Левобережная 1а', 'Мира 14', 'ЗЖБИиДК 1',
                    'Строителей 18', 'Панфиловская 33', 'Мастеркова 4']
    }
)
client_base

,client_id,address
0,111,Комсомольская 4
1,112,Энтузиастов 8а
2,113,Левобережная 1а
3,114,Мира 14
4,115,ЗЖБИиДК 1
5,116,Строителей 18
6,117,Панфиловская 33
7,118,Мастеркова 4


In [29]:
revenue_df = (
    rzd.merge(auto, on="client_id", how="outer")
    .merge(air, on="client_id", how="outer")
    .fillna(0)
)

In [30]:
df_revenue_with_address = client_base.merge(revenue_df, on="client_id", how="left").fillna(0)

In [31]:
revenue_df

,client_id,rzd_revenue,auto_revenue,air_revenue
0,111,1093.0,0.0,0.0
1,112,2810.0,0.0,0.0
2,113,10283.0,57483.0,0.0
3,114,5774.0,83.0,0.0
4,115,981.0,912.0,81.0
5,116,0.0,4834.0,4.0
6,117,0.0,98.0,13.0
7,118,0.0,0.0,173.0


In [ ]:
revenue_with_address_df